In [83]:
import numpy as np
import pandas as pd
from utils import rolling_apply


In [88]:
class self:
    period=4
data = pd.DataFrame()
data['date'] = ['01-01','02-01','03-01','04-01','05-01','06-01','07-01','08-01']
data['close'] = [1,2,3,4,7,10,20,30]
data = pd.DataFrame(data)
df = pd.DataFrame()
df['date'] = data['date']
df['close'] = data['close']
df['close_diff'] = df['close'].diff(periods=1)
df['gains'] = np.maximum(df['close_diff'], 0.0)
df['losses'] = np.maximum(-df['close_diff'], 0.0)
df.index = df.date
alpha = 1 - (self.period - 1)/self.period
avg_gain_df = rolling_apply(df, on='gains', window=self.period,
                               func=lambda x, previous: alpha * x + (1 - alpha) * previous,
                               init_func=lambda x: x.sum() / x.shape[0])
df['avg_gain'] = avg_gain_df['rolled']

avg_loss_df = rolling_apply(df, on='losses', window=self.period,
                               func=lambda x, previous: alpha * x + (1 - alpha) * previous,
                               init_func=lambda x: x.sum() / x.shape[0])

df['avg_loss'] = avg_loss_df['rolled']

df['rs'] = df['avg_gain'] / df['avg_loss']
df['rsi'] = 100 - (100 / (1 + df['rs']))

C:\Users\ugurg\Anaconda3\lib\site-packages\ipykernel_launcher.py:11: RuntimeWarning: invalid value encountered in maximum
  # This is added back by InteractiveShellApp.init_path()
C:\Users\ugurg\Anaconda3\lib\site-packages\ipykernel_launcher.py:12: RuntimeWarning: invalid value encountered in maximum
  if sys.path[0] == '':


In [89]:
df



,date,close,close_diff,gains,losses,avg_gain,avg_loss,rs,rsi
date,,,,,,,,,
01-01,01-01,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
02-01,02-01,2,1.0,1.0,0.0,NaN,NaN,NaN,NaN
03-01,03-01,3,1.0,1.0,0.0,NaN,NaN,NaN,NaN
04-01,04-01,4,1.0,1.0,0.0,0.750000,0.0,inf,100.0
05-01,05-01,7,3.0,3.0,0.0,1.312500,0.0,inf,100.0
06-01,06-01,10,3.0,3.0,0.0,1.734375,0.0,inf,100.0
07-01,07-01,20,10.0,10.0,0.0,3.800781,0.0,inf,100.0
08-01,08-01,30,10.0,10.0,0.0,5.350586,0.0,inf,100.0


In [17]:
def custom(x,i):
    print("x: ", x,"||index: ",i)


In [18]:
pd.rolling_apply(df['gains'], window=4, func=lambda x:custom(x,df['date']))

x:  [  0.   0.   0.  12.] ||index:  0    01-01
1    02-01
2    03-01
3    04-01
4    05-01
5    06-01
6    07-01
7    08-01
Name: date, dtype: object


C:\Users\ugurg\Anaconda3\lib\site-packages\ipykernel_launcher.py:1: FutureWarning: pd.rolling_apply is deprecated for Series and will be removed in a future version, replace with 
	Series.rolling(window=4,center=False).apply(func=<function>,args=<tuple>,kwargs=<dict>)
  """Entry point for launching an IPython kernel.


TypeError: must be real number, not NoneType

In [32]:
df.index = df.date

In [33]:
df.loc['01-01','close']

3

In [44]:
for idx,i in enumerate(df.index,1):
    print(idx,i,df.loc[i,'close'])

1 01-01 3
2 02-01 2
3 03-01 0
4 04-01 -5
5 05-01 7
6 06-01 10
7 07-01 -10
8 08-01 8


In [66]:
def my_rolling_apply(dataframe, on, window, func, init_func=None):

    new = pd.DataFrame(dataframe.index)  # create empty dataframe
    new.index = dataframe.index  # set index to general index - in our case 'date'
    new['orig'] = dataframe[on]
    new['rolled'] = [0] * new.shape[0]  # create new rolled column
    for enum, index in enumerate(dataframe.index):
        if enum < window - 1:  # not enough data to calculate
            new.loc[index, 'rolled'] = np.nan
        elif enum == window - 1:
            if init_func is None:
                new.loc[index, 'rolled'] = np.nan
            else:
                new.loc[index, 'rolled'] = init_func(dataframe[on].iloc[enum+1 - window:enum+1])
        else:
            new.loc[index, 'rolled'] = func(dataframe.loc[index,on], new['rolled'].iloc[enum - 1])

    return new

In [79]:
my_rolling_apply(df,on='gains',window=4, func=lambda x,prev: 0.25*x + (1-0.25)*prev, init_func=lambda x: x.sum()/x.shape[0])

,date,orig,rolled
date,,,
01-01,01-01,NaN,NaN
02-01,02-01,0.0,NaN
03-01,03-01,0.0,NaN
04-01,04-01,0.0,0.0000
05-01,05-01,12.0,3.0000
06-01,06-01,3.0,3.0000
07-01,07-01,0.0,2.2500
08-01,08-01,18.0,6.1875


In [78]:
my_rolling_apply(df,'losses',4, lambda x,prev: 0.25*x + (1-0.25)*prev,  lambda x: x.sum()/x.shape[0])

,date,orig,rolled
date,,,
01-01,01-01,NaN,NaN
02-01,02-01,1.0,NaN
03-01,03-01,2.0,NaN
04-01,04-01,5.0,2.000000
05-01,05-01,0.0,1.500000
06-01,06-01,0.0,1.125000
07-01,07-01,20.0,5.843750
08-01,08-01,0.0,4.382812


In [72]:
df

,date,close,close_diff,gains,losses
date,,,,,
01-01,01-01,3,NaN,NaN,NaN
02-01,02-01,2,-1.0,0.0,1.0
03-01,03-01,0,-2.0,0.0,2.0
04-01,04-01,-5,-5.0,0.0,5.0
05-01,05-01,7,12.0,12.0,0.0
06-01,06-01,10,3.0,3.0,0.0
07-01,07-01,-10,-20.0,0.0,20.0
08-01,08-01,8,18.0,18.0,0.0
